# 🚀 AI LVQ Std BOT — Monthly Specialist Fine-Tuning (`kingescaped/lvq-std-BOT`)

Notebook huấn luyện tinh chỉnh (**Fine-tune**) tự động hàng tháng trên **Kaggle GPU T4** cho 2 nhánh chuyên biệt trên repo Private Hugging Face **`kingescaped/lvq-std-BOT`**:
- **`btc-m15`**: Kéo tự động **1 năm (365 ngày)** nến M15 đã đóng của Bitcoin từ **Binance Public API** (`data-api.binance.vision`).
- **`xau-h1`**: Kéo tự động **2 năm (`730d`)** nến H1 đã đóng của Vàng từ **Yahoo Finance (`GC=F`)**.

### Thiết lập Kaggle Secrets 1 lần duy nhất:
Mở Notebook này ở chế độ **Edit → Add-ons → Secrets**, thêm và tích chọn gắn vào Notebook:
- `HF_TOKEN` *(bắt buộc)*: Token quyền **Write** tại `https://huggingface.co/settings/tokens`.
- `TELEGRAM_BOT_TOKEN` & `TELEGRAM_GROUP_ID` *(tùy chọn)*: Để nhận thông báo kết quả fine-tune hàng tháng qua Telegram.

In [ ]:
!pip install -q huggingface_hub safetensors einops tqdm yfinance "pandas>=2.0.0,<3.0.0" "six>=1.16.0" requests

import os
import subprocess

WORK_ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
REPO_DIR = os.path.join(WORK_ROOT, 'xauusd_signal_bot')
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/kingescaped/xauusd_signal_bot.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)

In [ ]:
import os
import torch


def _read_secret(secret_name):
    env_val = (os.getenv(secret_name) or '').strip()
    if env_val:
        return env_val
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(secret_name) or '').strip()
    except Exception:
        pass
    try:
        from google.colab import userdata
        return (userdata.get(secret_name) or '').strip()
    except Exception:
        return ''


HF_TOKEN = _read_secret('HF_TOKEN')
TELEGRAM_BOT_TOKEN = _read_secret('TELEGRAM_BOT_TOKEN')
TELEGRAM_GROUP_ID = _read_secret('TELEGRAM_GROUP_ID')
TARGET_REPO = 'kingescaped/lvq-std-BOT'

if TELEGRAM_BOT_TOKEN:
    os.environ['TELEGRAM_BOT_TOKEN'] = TELEGRAM_BOT_TOKEN
if TELEGRAM_GROUP_ID:
    os.environ['TELEGRAM_GROUP_ID'] = TELEGRAM_GROUP_ID

gpu_label = f'GPU ({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else 'CPU'
print('Thiết bị huấn luyện:', gpu_label)

READY_TO_TRAIN = bool(HF_TOKEN)
if not READY_TO_TRAIN:
    print(
        '[CẦN CẤU HÌNH 1 LẦN] Chưa tìm thấy HF_TOKEN trong Kaggle Secrets.\n'
        '-> Mở Notebook này trên Kaggle -> Edit -> Add-ons -> Secrets -> thêm HF_TOKEN (quyền Write) và tích chọn gắn vào Notebook!'
    )
    if TELEGRAM_BOT_TOKEN and TELEGRAM_GROUP_ID:
        from telegram_notifier import send_telegram_message
        send_telegram_message('⚠️ <b>[AI LVQ Std BOT]</b> Chưa tìm thấy <code>HF_TOKEN</code> trong Kaggle Secrets, bỏ qua lượt fine-tune tháng này.')
else:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from hf_repo_sync import sync_base_to_private_repo
    sync_info = sync_base_to_private_repo(target_repo=TARGET_REPO, token=HF_TOKEN)
    print('Trạng thái repo Private:', sync_info)

In [ ]:
btc_metrics = None
gold_metrics = None

if READY_TO_TRAIN:
    from finetune_data import build_kronos_datasets, fetch_binance_btc_m15, fetch_gold_h1_candles
    from finetune_trainer import FineTuneConfig, run_specialist_finetune

    btc_df = fetch_binance_btc_m15(days=365, symbol='BTCUSDT')
    print(f'Đã tải {len(btc_df):,} nến BTC M15 đã đóng ({btc_df["time"].iloc[0]} -> {btc_df["time"].iloc[-1]})')
    btc_train, btc_val = build_kronos_datasets(btc_df, context_len=200, pred_len=12, stride=4)
    btc_cfg = FineTuneConfig(symbol='BTCUSD', branch='btc-m15', context_len=200, pred_len=12, epochs=5, batch_size=16, learning_rate=1e-6, weight_decay=0.1)
    btc_metrics = run_specialist_finetune(btc_train, btc_val, btc_cfg, repo_id=TARGET_REPO, token=HF_TOKEN)
    print('Kết quả BTC:', btc_metrics)

    gold_df = fetch_gold_h1_candles(period='730d', ticker='GC=F')
    print(f'Đã tải {len(gold_df):,} nến Vàng H1 đã đóng ({gold_df["time"].iloc[0]} -> {gold_df["time"].iloc[-1]})')
    gold_train, gold_val = build_kronos_datasets(gold_df, context_len=200, pred_len=6, stride=2)
    gold_cfg = FineTuneConfig(symbol='XAUUSD', branch='xau-h1', context_len=200, pred_len=6, epochs=5, batch_size=16, learning_rate=1e-6, weight_decay=0.1)
    gold_metrics = run_specialist_finetune(gold_train, gold_val, gold_cfg, repo_id=TARGET_REPO, token=HF_TOKEN)
    print('Kết quả Vàng:', gold_metrics)

In [ ]:
if READY_TO_TRAIN and btc_metrics and gold_metrics:
    import kronos_predictor
    from telegram_notifier import send_telegram_message

    kronos_predictor._GLOBAL_PREDICTOR = None
    pred_btc = kronos_predictor.load_kronos_base('BTCUSD', 'btc-m15')
    pred_xau = kronos_predictor.load_kronos_base('XAUUSD', 'xau-h1')
    print('Nguồn model BTC:', getattr(pred_btc, 'model_source', None))
    print('Nguồn model XAU:', getattr(pred_xau, 'model_source', None))

    if TELEGRAM_BOT_TOKEN and TELEGRAM_GROUP_ID:
        btc_state = 'ĐÃ CẬP NHẬT' if btc_metrics['pushed'] else 'GIỮ BẢN CŨ'
        xau_state = 'ĐÃ CẬP NHẬT' if gold_metrics['pushed'] else 'GIỮ BẢN CŨ'
        msg = (
            '🧠 <b>[AI LVQ Std BOT] - HOÀN TẤT FINE-TUNE ĐỊNH KỲ THÁNG</b>\n'
            '━━━━━━━━━━━━━━━━━━━━\n'
            f'🟠 <b>BTC (btc-m15):</b> <code>{btc_metrics["baseline_val_loss"]:.4f} → {btc_metrics["best_val_loss"]:.4f}</code> ({btc_state})\n'
            f'🟡 <b>XAU (xau-h1):</b> <code>{gold_metrics["baseline_val_loss"]:.4f} → {gold_metrics["best_val_loss"]:.4f}</code> ({xau_state})\n'
            '━━━━━━━━━━━━━━━━━━━━\n'
            '✅ <i>Chu kỳ quét tiếp theo sẽ tự động nạp bộ trọng số mới nhất.</i>'
        )
        send_telegram_message(msg)